# Adversarial Detection — Pipeline Overview

Visualize clean vs adversarial images and run the trained detector.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import torch
from PIL import Image
import matplotlib.pyplot as plt

from detector.model import AdversarialDetector
from utils.preprocess import pil_to_tensor

CLEAN = ROOT / "data/clean"
ADV = ROOT / "data/adversarial"
MODEL_PATH = ROOT / "models/detector.pt"

In [ ]:
pairs = sorted(CLEAN.glob("sample_*.jpg"))[:3]
fig, axes = plt.subplots(len(pairs), 2, figsize=(8, 3 * len(pairs)))
if len(pairs) == 1:
    axes = [axes]
for ax_row, clean_path in zip(axes, pairs):
    adv_path = ADV / f"fgsm_{clean_path.name}"
    ax_row[0].imshow(Image.open(clean_path))
    ax_row[0].set_title(f"Clean: {clean_path.name}")
    ax_row[0].axis("off")
    ax_row[1].imshow(Image.open(adv_path))
    ax_row[1].set_title(f"FGSM: {adv_path.name}")
    ax_row[1].axis("off")
plt.tight_layout()

In [ ]:
device = torch.device("cpu")
model = AdversarialDetector().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

for path in [CLEAN / "sample_01.jpg", ADV / "fgsm_sample_01.jpg"]:
    t = pil_to_tensor(Image.open(path), 64, device)
    prob = torch.sigmoid(model(t)).item()
    label = "adversarial" if prob > 0.5 else "clean"
    print(f"{path.name:25s} -> {label} ({prob:.1%})")